# 02 — Monte Carlo uncertainty

Propagates the exchange-level uncertainty declared in `src/inventory.py`
(the foreground lognormals in `_SIGMA_LN`, plus ecoinvent's own background
pedigree distributions) through the full LCA for every scenario and the six
endpoint impact categories.

This notebook is a thin runnable wrapper over `monte_carlo.py` — the actual
engine lives there so the script and this notebook cannot drift. It reuses
`src/inventory.build_foreground_db`, `src/lca_setup` and the `Scenario`
objects, exactly like `01_scenarios.ipynb`.

**Independent of the deterministic pipeline.** It writes its own tables and
does not touch `all_scenarios.csv` or any figure; the plotting scripts keep
using the deterministic means.

**Outputs** (in `results/tables/`):
- `mc_scores_all_scenarios.csv` — long: scenario, category, iteration, score
- `mc_summary_all_scenarios.csv` — per (scenario, category): mean, median, sd, p2.5, p50, p97.5, CV%, n

**Runtime** ≈ 1 s/iteration across the six methods, so ~`6 × N` seconds per
scenario. N=1000 over five scenarios is an overnight (~8 h) job. Use the
`--test`-equivalent `N_ITERATIONS = 8` first to time it on your machine.

> Run `00_setup` and `01_scenarios` first (the project, methods and
> foreground builder must exist). Run this **after** any inventory change so
> the uncertainty is propagated around current central values.

Runs after `01_scenarios` (needs the Brightway project and scenario tables) and
before `03_results` (which draws the MC-based figures). The distribution plots
that used to live here moved to `03_results` / `src.plotting.make_si_mc_plots`.


In [ ]:
# Environment + imports
import os, sys, time
sys.path.insert(0, os.path.abspath('.'))

_prefix = sys.prefix
if os.name == 'nt':
    os.environ.setdefault('GDAL_DATA', os.path.join(_prefix, 'Library', 'share', 'gdal'))
    os.environ.setdefault('PROJ_LIB',  os.path.join(_prefix, 'Library', 'share', 'proj'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bw2data as bd

from src.config import PROJECT_NAME, RESULTS_TABLES_DIR
from src.scenarios import ALL_SCENARIOS
# The engine: reused verbatim so the notebook and monte_carlo.py stay in sync
from src.monte_carlo import run, _METHODS

bd.projects.set_current(PROJECT_NAME)
print('Project:', PROJECT_NAME)
print('Scenarios available:', [s.name for s in ALL_SCENARIOS])
print('Categories:', [m[0] for m in _METHODS])

## Configuration

Set the number of iterations, the RNG seed (fixed for reproducibility), and
which scenarios to run. Start with `N_ITERATIONS = 8` as a smoke test.

In [ ]:
# ── knobs ──────────────────────────────────────────────────────────────
N_ITERATIONS  = 8            # 8 = smoke test; 1000 = full run (~8 h all scenarios)
SEED          = 20240101     # fixed RNG seed for reproducibility
SCENARIO_NAMES = None        # None = all; or e.g. ['baseline', 'high_yield']
# ───────────────────────────────────────────────────────────────────────

scenarios = ALL_SCENARIOS if SCENARIO_NAMES is None else [
    s for s in ALL_SCENARIOS if s.name in set(SCENARIO_NAMES)
]
assert scenarios, f'No scenarios matched {SCENARIO_NAMES}'
OUT_TAG = '_test' if N_ITERATIONS <= 20 else ''   # keep smoke tests separate
print(f'{len(scenarios)} scenario(s) x {len(_METHODS)} methods x {N_ITERATIONS} iterations')
print('output tag:', repr(OUT_TAG))

## Run

Builds the foreground once per scenario, then resamples every uncertain
exchange `N_ITERATIONS` times per method. The `vary` column must read `Y`
for all six categories — `N` would mean the resampler is not reaching that
method (a warning is printed at the end if so). Progress prints per method.

In [ ]:
t0 = time.time()
long_df, summ_df = run(scenarios, N_ITERATIONS, SEED, out_tag=OUT_TAG)
print(f'\nWall time: {time.time()-t0:.0f} s')

## Summary

Median with 95% interval (2.5th–97.5th percentile) and CV per category. The
deterministic value is shown alongside; MC medians sitting above it for the
right-skewed (lognormal) categories is expected, not an error.

In [ ]:
pd.set_option('display.float_format', lambda v: f'{v:.3e}' if abs(v) < 1e-2 else f'{v:.3f}')
cols = ['scenario','category','deterministic','median','p2.5','p97.5','cv_pct','n','varied']
summ_df[cols].sort_values(['scenario','category'])

## Ecotoxicity stable-subset MC

Separate, resumable per-flow run that produces the ecotoxicity intervals used in
Figure 4 (`mc_ecotox_stable.csv`). Roughly 100 minutes for all scenarios; results
append per scenario, so a crash costs one scenario, not the run.


In [ ]:
RUN_ECOTOX_STABLE = False   # set True to (re)compute

if RUN_ECOTOX_STABLE:
    from src.mc_ecotox import run as run_ecotox
    from src.scenarios import ALL_SCENARIOS
    run_ecotox(ALL_SCENARIOS, n=N_ITERATIONS, seed=SEED, resume=True)
